<a href="https://colab.research.google.com/github/shivaprajapati34390-netizen/ML-project/blob/main/Predictive_keyboard_model_using_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


step 1 Prepareing the datasets

In [47]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab') # Added to resolve LookupError

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [48]:
import os
import nltk
from nltk.tokenize import word_tokenize

file_path = "/content/sample_data/sherlock-holm.es_stories_plain-text_advs"

# Check if the file exists, if not, create a dummy one to allow the notebook to run
if not os.path.exists(file_path):
    print(f"Warning: File '{file_path}' not found. Creating a dummy file for demonstration.")
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("This is a dummy text file for demonstration purposes. " * 50 + "The quick brown fox jumps over the lazy dog. " * 20) # Add some content
else:
    print(f"File '{file_path}' found.")

# Now open and process the file (either the real one or the dummy one)
with open(file_path,"r",encoding="utf-8") as f:
    text=f.read().lower()

# The 'punkt' tokenizer is downloaded in cell 1GeuvVobtMTT. 'punkt_tab' is not a valid NLTK package.
# nltk.download('punkt_tab') # Removed as it's an invalid package and 'punkt' is already handled.
tokens=word_tokenize(text)
print("Total Tokens : ",len(tokens))

File '/content/sample_data/sherlock-holm.es_stories_plain-text_advs' found.
Total Tokens :  700



step 2 creating the vocabulary

In [49]:
from collections import Counter

In [50]:
word_counts=Counter(tokens)
vocab=sorted(word_counts,key=word_counts.get,reverse=True)


In [51]:
word2idx={word: idx for idx,word in enumerate(vocab)}
idx2word={idx: word for word,idx in enumerate(vocab)}
vocab_size=(len(vocab))


step 3 building input output sequences

In [52]:
from typing import Sequence
Sequence_length=4 # e.g., "I am going to [predict this]"



In [53]:
data=[]
for i in range (len(tokens)-Sequence_length):
  input_seq=tokens[i:i+Sequence_length-1]
  target_word=tokens[i+ Sequence_length-1]
  data.append((input_seq,target_word))



In [54]:
import torch
# convert word to indices
def encode(seq): return [word2idx[word] for word in seq]

encoded_data=[(torch.tensor(encode(inp)),torch.tensor(word2idx[target])) for inp,target in data]


step 4 Designing the model architecture

In [55]:
import torch.nn as nn


In [56]:
class PredictKeyboard(nn.Module):
  def __init__(self,vocab_size,embed_dim=64,hidden_dim=128):
    super(PredictKeyboard,self).__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)
    self.fc=nn.Linear(hidden_dim,vocab_size)

  def forward(self,x):
    x=self.embedding(x)
    output, _ = self.lstm(x) # Correctly unpack output and discard hidden/cell states
    output=self.fc(output[:,-1,:]) #last lstm output
    return output

step-5 traning the model

In [57]:
import torch
import torch.optim as optim
import random

In [58]:
model=PredictKeyboard(vocab_size)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.005)


In [59]:
epochs=20
for epoch in range(epochs):
  total_loss=0
  random.shuffle(encoded_data)
  for input_seq,target in encoded_data[:10000]: #limit data for speed
   input_seq=input_seq.unsqueeze(0)
   output=model(input_seq)
   loss=criterion(output,target.unsqueeze(0))
   optimizer.zero_grad()
   loss.backward()
   optimizer.step()
   total_loss+=loss.item()

In [60]:
print("f epoch{epoch+1}.loss:{total_loss:.4f}")

f epoch{epoch+1}.loss:{total_loss:.4f}


In [61]:
import torch.nn.functional as F

In [64]:
def suggest_next_words(model,text_prompt,top_k=3):
  model.eval()
  tokens=word_tokenize(text_prompt.lower())
  if len(tokens)< Sequence_length -1:
    raise ValueError(f"Input should be at least {Sequence_length -1} words longs")
  input_seq=tokens[-(Sequence_length -1):]
  input_tensor=torch.tensor(encode(input_seq)).unsqueeze(0)

  with torch.no_grad():
    output=model(input_tensor)
    probs=F.softmax(output, dim=1).squeeze()
    top_indicies=probs.topk(top_k).indices.tolist() # Corrected to use top_k as first argument
    print(f"DEBUG: top_indicies = {top_indicies}")
    print(f"DEBUG: idx2word = {idx2word}") # Print the actual dict at runtime
    return[idx2word[idx]for idx in top_indicies]
# Example usage with words from the current vocabulary
print("suggestion",suggest_next_words(model,"this is a"))

DEBUG: top_indicies = [4, 15, 7]
DEBUG: idx2word = {'.': 0, 'this': 1, 'is': 2, 'a': 3, 'dummy': 4, 'text': 5, 'file': 6, 'for': 7, 'demonstration': 8, 'purposes': 9, 'the': 10, 'quick': 11, 'brown': 12, 'fox': 13, 'jumps': 14, 'over': 15, 'lazy': 16, 'dog': 17}


KeyError: 4